In [1]:
import sys
import torch
from src.language_models.model import RNNModel
import src.language_models.model as model_module
from src.language_models.dictionary_corpus import Dictionary

# Make the model module available for unpickling
sys.modules['model'] = model_module


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dictionary = Dictionary("data/base_data")  # Adjust path if needed


In [10]:
ntokens = len(dictionary)
emsize = 650
nhid = 650
nlayers = 2
dropout = 0.2
tied = False
model_type = "LSTM"


In [11]:
# Workaround for PyTorch version compatibility
import torch.nn as nn

# Save original __setstate__
_original_lstm_setstate = nn.LSTM.__setstate__

def _patched_lstm_setstate(self, d):
    # Initialize _flat_weights if missing
    if not hasattr(self, '_flat_weights'):
        self._flat_weights = []
    if not hasattr(self, '_flat_weights_names'):
        self._flat_weights_names = []
    # Now call the original
    try:
        _original_lstm_setstate(self, d)
    except AttributeError:
        # Fallback: manually reconstruct
        self.__dict__.update(d)
        self._flat_weights = []
        self._flat_weights_names = []
        self.flatten_parameters()

# Temporarily patch LSTM
nn.LSTM.__setstate__ = _patched_lstm_setstate

try:
    # Load checkpoint
    checkpoint = torch.load(r"C:\Users\malor\Documents\GTH\hidden650_batch128_dropout0.2_lr20.0.pt", 
                            map_location=device,
                            weights_only=False)
    
    print("Checkpoint type:", type(checkpoint))
    
    # Create model from scratch
    model = RNNModel(model_type, ntokens, emsize, nhid, nlayers, dropout, tied)
    
    # Load the state
    if isinstance(checkpoint, dict):
        model.load_state_dict(checkpoint, strict=False)
    else:
        # If checkpoint is the full model, extract state_dict
        model.load_state_dict(checkpoint.state_dict(), strict=False)
    
    model = model.to(device)
    model.eval()
    print("Model loaded successfully!")
    
finally:
    # Restore original __setstate__
    nn.LSTM.__setstate__ = _original_lstm_setstate

c:\Users\malor\Documents\GTH\gth\Lib\site-packages\torch\serialization.py:1749: SourceChangeWarning: source code of class 'src.language_models.model.RNNModel' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
c:\Users\malor\Documents\GTH\gth\Lib\site-packages\torch\serialization.py:1749: SourceChangeWarning: source code of class 'torch.nn.modules.dropout.Dropout' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
c:\Users\malor\Documents\GTH\gth\Lib\site-packages\torch\serialization.py:1749: SourceChangeWarning: source code of class 'torch.nn.modules.sparse.Embedding' has changed. you can retrieve the original source code by accessing the object's s

Checkpoint type: <class 'src.language_models.model.RNNModel'>
Model loaded successfully!


In [12]:
# Calculate perplexity per sentence (optimized with batching)
import numpy as np
import matplotlib.pyplot as plt
from torch.nn.utils.rnn import pad_sequence

def calculate_batch_perplexity(model, batch_sentences, device, ntokens):
    """Calculate perplexity for a batch of sentences"""
    # Pad sequences to same length
    padded = pad_sequence(batch_sentences, batch_first=False, padding_value=0)
    seq_len, batch_size = padded.shape
    
    if seq_len < 2:
        return [None] * batch_size
    
    # Initialize hidden state
    hidden = model.init_hidden(batch_size)
    
    criterion = torch.nn.CrossEntropyLoss(reduction='none')
    losses = torch.zeros(batch_size, device=device)
    counts = torch.zeros(batch_size, device=device)
    
    with torch.no_grad():
        for i in range(seq_len - 1):
            input_token = padded[i:i+1]  # (1, batch_size)
            target = padded[i+1]  # (batch_size)
            
            # Forward pass
            output, hidden = model(input_token, hidden)
            
            # Calculate loss for each item in batch
            loss = criterion(output.view(-1, ntokens), target.view(-1))
            
            # Only count non-padded positions
            mask = (target != 0).float()
            losses += loss * mask
            counts += mask
    
    # Calculate perplexities
    perplexities = []
    for i in range(batch_size):
        if counts[i] > 0:
            avg_loss = losses[i] / counts[i]
            ppl = np.exp(avg_loss.item())
            perplexities.append(ppl)
        else:
            perplexities.append(None)
    
    return perplexities

# Read valid.txt and split into sentences
print("Reading validation file...")
with open("data/base_data/valid.txt", 'r', encoding='utf8') as f:
    content = f.read()

# Split by words and find <eos> tokens
words = content.split()
eos_idx = dictionary.word2idx.get('<eos>', None)

print(f"Total words: {len(words)}")
print(f"<eos> token index: {eos_idx}")

# Split into sentences (sequences ending with <eos>)
sentences = []
current_sentence = []

for word in words:
    current_sentence.append(word)
    if word == '<eos>':
        sentences.append(current_sentence)
        current_sentence = []

if current_sentence:  # Add last sentence if it doesn't end with <eos>
    sentences.append(current_sentence)

print(f"Number of sentences: {len(sentences)}")
print(f"Sample sentence: {' '.join(sentences[0][:20])}...")

# Tokenize all sentences first
print("Tokenizing sentences...")
tokenized_sentences = []
for sentence in sentences:
    token_ids = []
    for word in sentence:
        if word in dictionary.word2idx:
            token_ids.append(dictionary.word2idx[word])
        else:
            token_ids.append(dictionary.word2idx.get("<unk>", 0))
    if len(token_ids) > 1:  # Need at least 2 tokens
        tokenized_sentences.append(torch.LongTensor(token_ids).to(device))

print(f"Tokenized {len(tokenized_sentences)} valid sentences")

# Calculate perplexity in batches
perplexities = []
sentence_lengths = []
batch_size = 128  # Process 128 sentences at a time

print("Calculating perplexities in batches...")
for start_idx in range(0, len(tokenized_sentences), batch_size):
    if start_idx % (batch_size * 10) == 0:
        print(f"Processing sentences {start_idx}/{len(tokenized_sentences)}")
    
    end_idx = min(start_idx + batch_size, len(tokenized_sentences))
    batch = tokenized_sentences[start_idx:end_idx]
    
    # Calculate perplexities for this batch
    batch_ppls = calculate_batch_perplexity(model, batch, device, ntokens)
    
    # Store results
    for i, ppl in enumerate(batch_ppls):
        if ppl is not None and not np.isnan(ppl) and not np.isinf(ppl):
            perplexities.append(ppl)
            sentence_lengths.append(len(batch[i]))

print(f"\nCalculated perplexity for {len(perplexities)} sentences")
print(f"Mean perplexity: {np.mean(perplexities):.2f}")
print(f"Median perplexity: {np.median(perplexities):.2f}")
print(f"Std perplexity: {np.std(perplexities):.2f}")

Reading validation file...
Total words: 10391172
<eos> token index: 19
Number of sentences: 381591
Sample sentence: So , a function " f " : " I " → R admits <unk> as a modulus of continuity...
Tokenizing sentences...
Tokenized 381591 valid sentences
Calculating perplexities in batches...
Processing sentences 0/381591
Processing sentences 1280/381591
Processing sentences 2560/381591
Processing sentences 3840/381591
Processing sentences 5120/381591
Processing sentences 6400/381591


KeyboardInterrupt: 

In [14]:
# Simple perplexity frequency plot (outliers removed)
import seaborn as sns

ppl = np.array(perplexities, dtype=float)
ppl = ppl[np.isfinite(ppl)]

# IQR outlier filter
q1, q3 = np.percentile(ppl, [25, 75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
ppl_filtered = ppl[(ppl >= lower) & (ppl <= upper)]

plt.figure(figsize=(10, 5))
sns.histplot(ppl_filtered, bins=60)
plt.xlabel('Perplexity')
plt.ylabel('Frequency')
plt.title('Sentence Perplexity Distribution (IQR outliers removed)')
plt.tight_layout()
plt.show()

print(f'Total sentences: {len(ppl)}')
print(f'After outlier removal: {len(ppl_filtered)}')
print(f'Removed as outliers: {len(ppl) - len(ppl_filtered)}')
print(f'Mean (filtered): {ppl_filtered.mean():.2f}')
print(f'Median (filtered): {np.median(ppl_filtered):.2f}')

ModuleNotFoundError: No module named 'seaborn'